In [ ]:
# 0 · Parámetros
EXP = "exp011_etiquetas_v2"
PADRE = "exp001"
SEED = 42
SMOKE = True

OPENAI_MODEL = "gpt-5-mini"
GPT_CONCURRENCY = 8
GPT_MAX_RETRIES = 5
GPT_MAX_OUTPUT_TOKENS = 4500
MAX_API_COST_USD = 40.0
# Tarifas estándar de gpt-5-mini por 1M tokens (consulta: 2026-10-08):
# https://developers.openai.com/api/docs/models/gpt-5-mini
OPENAI_INPUT_USD_PER_M = 0.25
OPENAI_CACHED_INPUT_USD_PER_M = 0.025
OPENAI_OUTPUT_USD_PER_M = 2.00

CANDIDATE_IDS = [f"L{i}" for i in range(9)]
PUBLIC_DATASETS = {
    "L2": "stevenleehans/rsna-knee-llm-report-labels",
    "L3": "pilkwang/rsna-knee-llm-labels",
    "L4": "lixin73/rsna-knee-llm-report-labels-sol56",
    "L5": "dreaddevelopment/rsna-knee-labels",
    "L6": "laymond/rsna-knee-abnormality-qwen3-8b-weak-labels",
}
PUBLIC_FILES = {
    "L2": "llm_labels_v4_blend.csv",
    "L3": "report_labels_v2.csv",
    "L4": "labels_llm_gpt56sol.csv",
    "L5": "labels_llm_soft.csv",
    "L6": "weak_labels_v2_filled.csv",
}
DATA_ROOT_OVERRIDE = None
RESULTS_ROOT_OVERRIDE = None
DEV_RELATIVE_PATH = "labels/dev_informes_v1.csv"
GPT_CACHE_RELATIVE_PATH = "labels/exp011_gpt_cache"
OUTPUT_RELATIVE_PATH = "labels/report_labels_v2.csv"

In [ ]:
# 1 · Setup
import hashlib
import json
import os
import random
import re
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import roc_auc_score

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(repo))
from src.knee.config import LABELS
from src.knee.experiments import guardar_metricas
from src.knee.splits import GOLD_FOLD, load_split

def load_secrets_env(path):
    path = Path(path)
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key, value = key.strip(), value.strip()
        if value[:1] == value[-1:] and value[:1] in {"'", '"'}:
            value = value[1:-1]
        os.environ.setdefault(key, value)

load_secrets_env(repo / ".secrets.env")  # nunca imprime secretos
random.seed(SEED)
np.random.seed(SEED)

data_setting = DATA_ROOT_OVERRIDE or os.environ.get("RSNA_DATA", "/workspace/data")
results_setting = RESULTS_ROOT_OVERRIDE or os.environ.get("RSNA_RESULTS", str(repo / "results"))
DATA_ROOT = Path(data_setting)
RAW_ROOT = DATA_ROOT / "raw"
LABELS_ROOT = DATA_ROOT / "labels"
PUBLIC_ROOT = LABELS_ROOT / "public"
DEV_PATH = DATA_ROOT / DEV_RELATIVE_PATH
GPT_CACHE = DATA_ROOT / GPT_CACHE_RELATIVE_PATH
OUTPUT_PATH = DATA_ROOT / OUTPUT_RELATIVE_PATH
RESULTS = Path(results_setting) / EXP
for path in (LABELS_ROOT, PUBLIC_ROOT, GPT_CACHE, RESULTS):
    path.mkdir(parents=True, exist_ok=True)

spec_path = repo / "experiments" / EXP / "SPEC.md"
if not spec_path.exists():
    raise FileNotFoundError(f"Falta la SPEC confirmada: {spec_path}")

train = pd.read_csv(RAW_ROOT / "train.csv", encoding="utf-8", low_memory=False)
train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
if len(LABELS) != 12 or not set(LABELS).issubset(train.columns):
    raise ValueError("train.csv no contiene las 12 etiquetas esperadas")
split = load_split()
split["StudyInstanceUID"] = split["StudyInstanceUID"].astype(str)
if not (split.loc[split.is_gold == 1, "fold"] == GOLD_FOLD).all():
    raise RuntimeError("El gold no está aislado en fold -1")
study_meta_path = LABELS_ROOT / "study_meta_v2.csv"
study_meta = pd.read_csv(study_meta_path, low_memory=False) if study_meta_path.exists() else pd.DataFrame()
print(f"{EXP}: {len(train):,} estudios · {len(LABELS)} etiquetas · SMOKE={SMOKE}")
print(f"DATA_ROOT={DATA_ROOT}")

In [ ]:
# 2 · Descargar, inspeccionar y armonizar L0–L6
CANDIDATE_NOTES_MARKDOWN = r"""
### Notas de procedencia de los candidatos

- **L0 — v1 local:** Qwen2.5-7B-Instruct. No se usó gold para generar; se evaluó contra gold después.
- **L1 — v1, silencio→0:** v1 guarda `0.5` tanto para *uncertain* como para *not mentioned*, sin un campo que los distinga. Por eso esta aproximación convierte **todo** `0.5` a `0`; no puede preservar los inciertos.
- **L2 — stevenleehans:** el modelo base no se identifica en la descripción. Se usa el archivo recomendado `llm_labels_v4_blend.csv` (v2 + GPT-5.6-Sol). La descripción sí declara validación y ajustes de Synovitis sobre los 58 gold; riesgo de afinado con gold: **sí**.
- **L3 — pilkwang:** el generador incluido usa `claude-opus-5` por defecto y devuelve scores YES/NO/UNK. La descripción no menciona uso de gold; riesgo declarado: **no indicado**.
- **L4 — lixin73:** GPT-5.6-Sol, según el nombre y la ficha del dataset. La descripción está vacía y no menciona uso de gold; riesgo declarado: **no indicado**.
- **L5 — dreaddevelopment:** LLM no especificado, probabilidades blandas. La descripción no menciona uso de gold; riesgo declarado: **no indicado**.
- **L6 — laymond:** Qwen2.5-32B-Instruct 4-bit; se usa `weak_labels_v2_filled.csv` y su `__confidence` como P(present). La descripción dice que prompt y reglas de coocurrencia se auditaron contra los 58 gold; riesgo de afinado con gold: **sí**.
"""
display(Markdown(CANDIDATE_NOTES_MARKDOWN))

def public_path(candidate_id):
    ref = PUBLIC_DATASETS[candidate_id]
    return PUBLIC_ROOT.joinpath(*ref.split("/"), PUBLIC_FILES[candidate_id])

def ensure_public_dataset(candidate_id):
    path = public_path(candidate_id)
    if path.exists():
        return path
    if not os.environ.get("KAGGLE_API_TOKEN"):
        raise RuntimeError(f"Falta KAGGLE_API_TOKEN para descargar {PUBLIC_DATASETS[candidate_id]}")
    destination = path.parent
    destination.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-m", "kaggle.cli", "datasets", "download",
               "-d", PUBLIC_DATASETS[candidate_id], "-p", str(destination), "--unzip"]
    subprocess.run(command, check=True)
    if not path.exists():
        raise FileNotFoundError(f"La descarga no produjo {path.name}")
    return path

def standardize_candidate(frame, candidate_id):
    required = ["StudyInstanceUID", *LABELS]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"{candidate_id}: faltan columnas {missing}")
    out = frame[required].copy()
    out["StudyInstanceUID"] = out["StudyInstanceUID"].astype(str)
    if out["StudyInstanceUID"].duplicated().any():
        raise ValueError(f"{candidate_id}: StudyInstanceUID duplicados")
    for label in LABELS:
        out[label] = pd.to_numeric(out[label], errors="coerce")
        invalid = out[label].notna() & ~out[label].between(0.0, 1.0)
        if invalid.any():
            raise ValueError(f"{candidate_id}: {label} tiene valores fuera de [0, 1]")
    return out

def harmonize_l0():
    return standardize_candidate(pd.read_csv(LABELS_ROOT / "report_labels_v1.csv", low_memory=False), "L0")

def harmonize_l1():
    out = harmonize_l0()
    out[LABELS] = out[LABELS].replace(0.5, 0.0)
    return standardize_candidate(out, "L1")

def harmonize_l2():
    return standardize_candidate(pd.read_csv(ensure_public_dataset("L2"), low_memory=False), "L2")

def harmonize_l3():
    return standardize_candidate(pd.read_csv(ensure_public_dataset("L3"), low_memory=False), "L3")

def harmonize_l4():
    return standardize_candidate(pd.read_csv(ensure_public_dataset("L4"), low_memory=False), "L4")

def harmonize_l5():
    return standardize_candidate(pd.read_csv(ensure_public_dataset("L5"), low_memory=False), "L5")

def harmonize_l6():
    raw = pd.read_csv(ensure_public_dataset("L6"), low_memory=False)
    mapped = pd.DataFrame({"StudyInstanceUID": raw["StudyInstanceUID"].astype(str)})
    for label in LABELS:
        mapped[label] = pd.to_numeric(raw[f"{label}__confidence"], errors="coerce")
    return standardize_candidate(mapped, "L6")

harmonizers = {
    "L0": harmonize_l0, "L1": harmonize_l1, "L2": harmonize_l2,
    "L3": harmonize_l3, "L4": harmonize_l4, "L5": harmonize_l5, "L6": harmonize_l6,
}
candidates, candidate_errors = {}, {}
for candidate_id, harmonizer in harmonizers.items():
    try:
        candidates[candidate_id] = harmonizer()
    except Exception as exc:
        candidate_errors[candidate_id] = f"{type(exc).__name__}: {exc}"
        candidates[candidate_id] = pd.DataFrame(columns=["StudyInstanceUID", *LABELS])
        print(f"{candidate_id} no disponible: {candidate_errors[candidate_id]}")

train_uids = set(train["StudyInstanceUID"])
def candidate_coverage(candidate_id):
    frame = candidates[candidate_id]
    if frame.empty:
        return 0, 0.0
    usable = frame[frame["StudyInstanceUID"].isin(train_uids)].set_index("StudyInstanceUID")[LABELS].notna().all(axis=1)
    n_complete = int(usable.sum())
    return n_complete, 100.0 * n_complete / len(train)

coverage_rows = []
for candidate_id in harmonizers:
    n_complete, coverage_pct = candidate_coverage(candidate_id)
    coverage_rows.append({"candidate": candidate_id, "complete_studies": n_complete, "coverage_pct": coverage_pct})
coverage_table = pd.DataFrame(coverage_rows)
display(coverage_table.style.format({"coverage_pct": "{:.2f}%"}))

In [ ]:
# 3 · L7: GPT por API, una llamada por informe y cache reanudable
OFFICIAL_DEFINITIONS_SOURCE = "https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/discussion/733343"
LABEL_DEFINITIONS = {
    "ACL": "Rotura parcial de alto grado o completa del ligamento cruzado anterior: discontinuidad completa o más del 50 % de fibras rotas. Cambio leve de señal, degeneración o engrosamiento sin discontinuidad es negativo.",
    "MCL": "Rotura aguda parcial de alto grado o completa del ligamento colateral medial, con fibras interrumpidas y edema dentro y alrededor del ligamento. Esguinces leves o cambios crónicos son negativos.",
    "Medial Meniscus": "Señal anormal que contacta definitivamente la superficie del menisco medial en al menos dos imágenes, o fragmento truncado, pequeño o desplazado. Degeneración intrameniscal sin llegar a la superficie es negativa.",
    "Lateral Meniscus": "Los mismos criterios de rotura meniscal, aplicados al menisco lateral.",
    "Medial OA": "Área moderada o grande (aprox. 1 cm o más) de pérdida de cartílago de alto grado, mayor del 50 % del grosor, en el compartimento medial, con o sin cambios subcondrales.",
    "Lateral OA": "Los mismos criterios de artrosis, aplicados al compartimento tibiofemoral lateral.",
    "PF OA": "Los mismos criterios de artrosis, aplicados al compartimento patelofemoral.",
    "Effusion": "Cantidad moderada o grande de líquido que distiende la articulación.",
    "Synovitis": "Inflamación y engrosamiento del revestimiento sinovial de la articulación.",
    "Baker's": "Colección líquida moderada o grande en la localización característica posterior de un quiste de Baker o poplíteo.",
    "Contusion": "Contusión ósea: señal tipo edema de médula ósea por impacto, sin una línea de fractura definida.",
    "Fracture": "Fractura aguda con interrupción cortical o línea de fractura.",
}
definitions_text = "\n".join(f"- {label}: {definition}" for label, definition in LABEL_DEFINITIONS.items())
L7_INSTRUCTIONS = f"""You are an expert musculoskeletal radiologist extracting structured findings from a knee MRI report in any language.
Use only the report text; do not infer a finding from symptoms, another finding, or outside knowledge.
Classify every label as present, absent, uncertain, or not_mentioned.
- present: the finding is explicitly present.
- absent: the report explicitly denies it or calls the structure normal/intact.
- uncertain: the report says possible, equivocal, suspicious, or cannot exclude.
- not_mentioned: the report gives no information about that finding.
For present, absent, and uncertain, copy the shortest verbatim evidence quote from the report.
For not_mentioned, return an empty evidence string. Never translate or paraphrase evidence.
Also report the knee side as left, right, bilateral, or unknown.

Official label definitions:
{definitions_text}
"""
label_schema = {
    "type": "object",
    "properties": {
        "status": {"type": "string", "enum": ["present", "absent", "uncertain", "not_mentioned"]},
        "evidence": {"type": "string"},
    },
    "required": ["status", "evidence"],
    "additionalProperties": False,
}
L7_SCHEMA = {
    "type": "object",
    "properties": {
        "knee_side": {"type": "string", "enum": ["left", "right", "bilateral", "unknown"]},
        "labels": {
            "type": "object",
            "properties": {label: label_schema for label in LABELS},
            "required": LABELS,
            "additionalProperties": False,
        },
    },
    "required": ["knee_side", "labels"],
    "additionalProperties": False,
}
STATUS_TO_VALUE = {"present": 1.0, "absent": 0.0, "uncertain": 0.5, "not_mentioned": 0.0}
PROMPT_VERSION = "exp011_l7_v2"
PROMPT_FINGERPRINT = hashlib.sha256(json.dumps(
    {"instructions": L7_INSTRUCTIONS, "schema": L7_SCHEMA},
    ensure_ascii=False, sort_keys=True,
).encode("utf-8")).hexdigest()

def cache_matches(record):
    return (record.get("model") == OPENAI_MODEL
            and record.get("prompt_version") == PROMPT_VERSION
            and record.get("prompt_fingerprint") == PROMPT_FINGERPRINT)

def normalize_evidence(text):
    return re.sub(r"\s+", " ", str(text)).strip().casefold()

def validate_l7_payload(payload, report):
    if payload.get("knee_side") not in {"left", "right", "bilateral", "unknown"}:
        raise ValueError("knee_side inválido")
    labels_payload = payload.get("labels", {})
    if set(labels_payload) != set(LABELS):
        raise ValueError("La respuesta no contiene exactamente las 12 etiquetas")
    report_normalized = normalize_evidence(report)
    hallucinations = {}
    for label in LABELS:
        item = labels_payload[label]
        status, evidence = item.get("status"), item.get("evidence")
        if status not in STATUS_TO_VALUE or not isinstance(evidence, str):
            raise ValueError(f"Salida inválida para {label}")
        quote = normalize_evidence(evidence)
        missing_required_quote = status != "not_mentioned" and not quote
        unexpected_silence_quote = status == "not_mentioned" and bool(quote)
        quote_not_found = bool(quote) and quote not in report_normalized
        hallucinations[label] = bool(missing_required_quote or unexpected_silence_quote or quote_not_found)
    return hallucinations

def response_output_text(response_payload):
    for item in response_payload.get("output", []):
        if item.get("type") != "message":
            continue
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                return content.get("text", "")
    raise ValueError("La API no devolvió output_text")

def model_supports_temperature_zero(model):
    # Los GPT-5 originales (incluido gpt-5-mini) rechazan temperature.
    return re.match(r"^gpt-5(?:$|-(?:mini|nano|pro)(?:-|$))", model) is None

def usage_and_cost(response_payload):
    usage = response_payload.get("usage", {})
    input_tokens = int(usage.get("input_tokens", 0) or 0)
    output_tokens = int(usage.get("output_tokens", 0) or 0)
    cached_tokens = int((usage.get("input_tokens_details") or {}).get("cached_tokens", 0) or 0)
    uncached_tokens = max(0, input_tokens - cached_tokens)
    cost = (uncached_tokens * OPENAI_INPUT_USD_PER_M + cached_tokens * OPENAI_CACHED_INPUT_USD_PER_M
            + output_tokens * OPENAI_OUTPUT_USD_PER_M) / 1_000_000
    return {"input_tokens": input_tokens, "cached_input_tokens": cached_tokens,
            "output_tokens": output_tokens, "cost_usd": cost}

usage_lock = threading.Lock()
l7_usage = {"input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
l7_budget = {"reserved_cost_usd": 0.0}
def add_usage(usage):
    with usage_lock:
        for key in l7_usage:
            l7_usage[key] += usage.get(key, 0)

def merge_usage(*items):
    merged = {key: 0 for key in l7_usage}
    for item in items:
        for key in merged:
            merged[key] += (item or {}).get(key, 0)
    return merged

def reserve_request_budget(body):
    # Cota conservadora: un token no puede representar menos de un byte del payload UTF-8.
    max_input_tokens = len(json.dumps(body, ensure_ascii=False).encode("utf-8"))
    worst_case = (max_input_tokens * OPENAI_INPUT_USD_PER_M
                  + GPT_MAX_OUTPUT_TOKENS * OPENAI_OUTPUT_USD_PER_M) / 1_000_000
    with usage_lock:
        committed = l7_usage["cost_usd"]
        reserved = l7_budget["reserved_cost_usd"]
        if committed + reserved + worst_case > MAX_API_COST_USD:
            raise RuntimeError(f"Presupuesto API reservado: US${committed + reserved:.2f}; "
                               f"la siguiente llamada podría superar US${MAX_API_COST_USD:.2f}")
        l7_budget["reserved_cost_usd"] += worst_case
    return worst_case

def release_request_budget(amount):
    with usage_lock:
        l7_budget["reserved_cost_usd"] = max(0.0, l7_budget["reserved_cost_usd"] - amount)

def known_side(uid):
    if study_meta.empty or not {"study", "study_side"}.issubset(study_meta.columns):
        return "unknown"
    match = study_meta.loc[study_meta["study"].astype(str) == str(uid), "study_side"]
    if match.empty:
        return "unknown"
    return {"L": "left", "R": "right", "B": "bilateral"}.get(str(match.iloc[0]).upper(), "unknown")

def call_openai_report(uid, report):
    body = {
        "model": OPENAI_MODEL,
        "instructions": L7_INSTRUCTIONS,
        "input": f"Known metadata side: {known_side(uid)}\n\nMRI REPORT:\n{report}",
        "text": {"format": {"type": "json_schema", "name": "knee_report_labels",
                            "schema": L7_SCHEMA, "strict": True}},
        "max_output_tokens": GPT_MAX_OUTPUT_TOKENS,
        "store": False,
    }
    if model_supports_temperature_zero(OPENAI_MODEL):
        body["temperature"] = 0
    reservation = reserve_request_budget(body)
    request = urllib.request.Request(
        "https://api.openai.com/v1/responses",
        data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}", "Content-Type": "application/json"},
        method="POST",
    )
    usage = None
    try:
        try:
            with urllib.request.urlopen(request, timeout=240) as response:
                response_payload = json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")[:1000]
            raise RuntimeError(f"OpenAI HTTP {exc.code}: {detail}") from exc
        usage = usage_and_cost(response_payload)
        add_usage(usage)  # cuenta también respuestas facturadas que luego fallen validación
        payload = json.loads(response_output_text(response_payload))
        hallucinations = validate_l7_payload(payload, report)
        return payload, hallucinations, usage
    except Exception as exc:
        if usage is None:
            usage = {"input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0,
                     "cost_usd": reservation, "estimated_from_failed_request": True}
            add_usage(usage)
        exc.usage = usage
        raise
    finally:
        release_request_budget(reservation)

def cache_path(uid):
    return GPT_CACHE / f"{uid}.json"

def write_cache_record(path, record):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

def process_l7_report(uid, report):
    path = cache_path(uid)
    previous_usage = merge_usage()
    previous_errors = []
    if path.exists():
        cached = json.loads(path.read_text(encoding="utf-8"))
        if cache_matches(cached) and cached.get("status") == "complete":
            cached["hallucinations"] = validate_l7_payload(cached["payload"], report)
            add_usage(cached.get("usage", {}))
            return cached
        if cache_matches(cached) and cached.get("status") == "error":
            previous_usage = merge_usage(cached.get("usage", {}))
            previous_errors = list(cached.get("errors", []))
            add_usage(previous_usage)
    attempt_usages = []
    for attempt in range(GPT_MAX_RETRIES):
        try:
            payload, hallucinations, usage = call_openai_report(uid, report)
        except Exception as exc:
            attempt_usages.append(getattr(exc, "usage", {}))
            previous_errors.append(f"attempt {attempt + 1}: {type(exc).__name__}: {exc}"[:1000])
            error_record = {
                "status": "error", "StudyInstanceUID": str(uid), "model": OPENAI_MODEL,
                "prompt_version": PROMPT_VERSION, "prompt_fingerprint": PROMPT_FINGERPRINT,
                "updated_at": datetime.now(timezone.utc).isoformat(),
                "usage": merge_usage(previous_usage, *attempt_usages), "errors": previous_errors,
            }
            write_cache_record(path, error_record)
            if attempt + 1 == GPT_MAX_RETRIES:
                raise
            time.sleep(min(30.0, 2.0 ** attempt) + random.random())
            continue
        record = {
            "status": "complete", "StudyInstanceUID": str(uid), "model": OPENAI_MODEL,
            "prompt_version": PROMPT_VERSION, "prompt_fingerprint": PROMPT_FINGERPRINT,
            "created_at": datetime.now(timezone.utc).isoformat(), "payload": payload,
            "hallucinations": hallucinations,
            "usage": merge_usage(previous_usage, *attempt_usages, usage),
            "errors": previous_errors,
            "temperature_sent": 0 if model_supports_temperature_zero(OPENAI_MODEL) else None,
        }
        write_cache_record(path, record)
        return record

l7_records, l7_errors = [], {}
gpt_source = train[["StudyInstanceUID", "Report"]].dropna(subset=["Report"]).copy()
if SMOKE:
    if DEV_PATH.exists():
        smoke_uids = set(pd.read_csv(DEV_PATH, usecols=["StudyInstanceUID"], dtype=str)["StudyInstanceUID"])
        gpt_source = gpt_source[gpt_source.StudyInstanceUID.isin(smoke_uids)]
    gpt_source = gpt_source.sample(n=min(20, len(gpt_source)), random_state=SEED)

pending_l7 = []
has_openai_key = bool(os.environ.get("OPENAI_API_KEY"))
for row in gpt_source.itertuples(index=False):
    path = cache_path(row.StudyInstanceUID)
    cached = json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
    if cached and cache_matches(cached) and cached.get("status") == "complete":
        cached["hallucinations"] = validate_l7_payload(cached["payload"], str(row.Report))
        add_usage(cached.get("usage", {}))
        l7_records.append(cached)
    else:
        if cached and cache_matches(cached) and cached.get("status") == "error" and not has_openai_key:
            add_usage(cached.get("usage", {}))
        pending_l7.append((str(row.StudyInstanceUID), str(row.Report)))

l7_pending_without_key = 0
if pending_l7 and not has_openai_key:
    l7_pending_without_key = len(pending_l7)
    print(f"OPENAI_API_KEY no disponible: {l7_pending_without_key} informes L7 pendientes; se conserva el cache válido.")
elif pending_l7:
    with ThreadPoolExecutor(max_workers=GPT_CONCURRENCY) as executor:
        futures = {executor.submit(process_l7_report, uid, report): uid for uid, report in pending_l7}
        for future in as_completed(futures):
            uid = futures[future]
            try:
                l7_records.append(future.result())
            except Exception as exc:
                l7_errors[uid] = f"{type(exc).__name__}: {exc}"
print(f"L7: {len(l7_records)} cacheados/completados · {len(l7_errors)} errores")

l7_rows, l7_hallucination_count = [], 0
for record in l7_records:
    row = {"StudyInstanceUID": record["StudyInstanceUID"]}
    for label in LABELS:
        hallucinated = bool(record["hallucinations"].get(label, False))
        l7_hallucination_count += int(hallucinated)
        status = record["payload"]["labels"][label]["status"]
        row[label] = np.nan if hallucinated else STATUS_TO_VALUE[status]
    l7_rows.append(row)
candidates["L7"] = standardize_candidate(pd.DataFrame(l7_rows, columns=["StudyInstanceUID", *LABELS]), "L7")
n_complete, coverage_pct = candidate_coverage("L7")
coverage_table = pd.concat([coverage_table, pd.DataFrame([{"candidate": "L7", "complete_studies": n_complete,
                                                            "coverage_pct": coverage_pct}])], ignore_index=True)
l7_hallucination_rate = (l7_hallucination_count / (12 * len(l7_records))) if l7_records else np.nan
print(f"L7 tokens in/out: {l7_usage['input_tokens']:,}/{l7_usage['output_tokens']:,} · costo US${l7_usage['cost_usd']:.4f}")

In [ ]:
# 4 · Evaluación contra dev; gold solo veta
def score_candidate(candidate_id, truth, min_positives=0, split_name="dev"):
    pred = candidates[candidate_id]
    merged = truth[["StudyInstanceUID", *LABELS]].merge(
        pred, on="StudyInstanceUID", how="left", suffixes=("_true", "_pred"))
    rows = []
    for label in LABELS:
        y_true_all = pd.to_numeric(truth[label], errors="coerce")
        n_positive = int((y_true_all == 1).sum())
        y_true = pd.to_numeric(merged[f"{label}_true"], errors="coerce")
        y_score = pd.to_numeric(merged[f"{label}_pred"], errors="coerce")
        valid = y_true.isin([0.0, 1.0]) & y_score.notna()
        auc = np.nan
        if valid.any() and y_true[valid].nunique() == 2:
            auc = float(roc_auc_score(y_true[valid], y_score[valid]))
        include = n_positive >= min_positives and np.isfinite(auc)
        rows.append({"candidate": candidate_id, "split": split_name, "label": label,
                     "n_valid": int(valid.sum()), "n_positive": n_positive,
                     "auc": auc, "in_macro": bool(include)})
    detail = pd.DataFrame(rows)
    included = detail.loc[detail.in_macro, "auc"]
    macro = float(included.mean()) if len(included) else np.nan
    return macro, detail

gold_uids = set(split.loc[split.fold == GOLD_FOLD, "StudyInstanceUID"])
gold = train[train["StudyInstanceUID"].isin(gold_uids)][["StudyInstanceUID", *LABELS]].copy()
if len(gold) != 58:
    raise RuntimeError(f"Se esperaban 58 estudios gold y se encontraron {len(gold)}")
for label in LABELS:
    gold[label] = pd.to_numeric(gold[label], errors="coerce")

dev_available = DEV_PATH.exists()
dev_schema = None
if dev_available:
    raw_dev = pd.read_csv(DEV_PATH, low_memory=False)
    numeric_columns = [f"{label}_numeric" for label in LABELS]
    if {"StudyInstanceUID", *numeric_columns}.issubset(raw_dev.columns):
        dev = raw_dev[["StudyInstanceUID", *numeric_columns]].rename(
            columns={f"{label}_numeric": label for label in LABELS})
        dev_schema = "exp010_status_plus_numeric"
    elif {"StudyInstanceUID", *LABELS}.issubset(raw_dev.columns):
        dev = raw_dev[["StudyInstanceUID", *LABELS]].copy()
        status_map = {"present": 1.0, "absent": 0.0, "not_mentioned": 0.0, "uncertain": np.nan}
        for label in LABELS:
            raw_values = dev[label]
            numeric_values = pd.to_numeric(raw_values, errors="coerce")
            normalized_status = raw_values.astype(str).str.strip().str.lower()
            mapped_values = normalized_status.map(status_map)
            unknown = (raw_values.notna() & numeric_values.isna()
                       & ~normalized_status.isin(status_map) & normalized_status.ne(""))
            if unknown.any():
                raise ValueError(f"dev: estados desconocidos en {label}: {sorted(raw_values[unknown].astype(str).unique())}")
            dev[label] = numeric_values.where(numeric_values.notna(), mapped_values)
        dev_schema = "numeric_or_status_columns"
    else:
        raise ValueError("dev_informes_v1.csv no tiene las columnas numéricas ni de estado esperadas")
    dev["StudyInstanceUID"] = dev["StudyInstanceUID"].astype(str)
    if dev.StudyInstanceUID.duplicated().any():
        raise ValueError("dev_informes_v1.csv contiene StudyInstanceUID duplicados")
    for label in LABELS:
        dev[label] = pd.to_numeric(dev[label], errors="coerce")
        invalid = dev[label].notna() & ~dev[label].isin([0.0, 1.0])
        if invalid.any():
            raise ValueError(f"dev: {label} contiene valores distintos de 0/1/NaN")
    print(f"dev cargado: {len(dev)} estudios · schema={dev_schema}")
else:
    dev = pd.DataFrame(columns=["StudyInstanceUID", *LABELS])
    print("dev pendiente")

coverage_map = coverage_table.set_index("candidate")["coverage_pct"].to_dict()
evaluation_rows, per_label_frames = [], []
for candidate_id in [f"L{i}" for i in range(8)]:
    gold_macro, gold_detail = score_candidate(candidate_id, gold, min_positives=1, split_name="gold")
    if dev_available:
        dev_macro, dev_detail = score_candidate(candidate_id, dev, min_positives=8, split_name="dev")
        per_label_frames.append(dev_detail)
    else:
        dev_macro = np.nan
    evaluation_rows.append({"candidate": candidate_id, "dev_macro_auc": dev_macro,
                            "gold_macro_auc": gold_macro, "coverage_pct": coverage_map.get(candidate_id, 0.0)})

l8_members = []
if dev_available:
    base_evaluation = pd.DataFrame(evaluation_rows)
    eligible_for_consensus = base_evaluation[(base_evaluation.coverage_pct >= 99.0) & base_evaluation.dev_macro_auc.notna()]
    l8_members = eligible_for_consensus.sort_values(["dev_macro_auc", "candidate"], ascending=[False, True]).head(3)["candidate"].tolist()
    if len(l8_members) == 3:
        uid_order = train["StudyInstanceUID"]
        aligned = [candidates[cid].set_index("StudyInstanceUID")[LABELS].reindex(uid_order) for cid in l8_members]
        consensus_values = sum(frame for frame in aligned) / len(aligned)
        consensus_values.index = uid_order.values
        consensus_values.index.name = "StudyInstanceUID"
        candidates["L8"] = standardize_candidate(consensus_values.reset_index(), "L8")
        n_complete, coverage_pct = candidate_coverage("L8")
        coverage_table = pd.concat([coverage_table, pd.DataFrame([{"candidate": "L8",
            "complete_studies": n_complete, "coverage_pct": coverage_pct}])], ignore_index=True)
        coverage_map["L8"] = coverage_pct
        dev_macro, dev_detail = score_candidate("L8", dev, min_positives=8, split_name="dev")
        gold_macro, _ = score_candidate("L8", gold, min_positives=1, split_name="gold")
        per_label_frames.append(dev_detail)
        evaluation_rows.append({"candidate": "L8", "dev_macro_auc": dev_macro,
                                "gold_macro_auc": gold_macro, "coverage_pct": coverage_pct})

evaluation_table = pd.DataFrame(evaluation_rows)
l0_gold = evaluation_table.loc[evaluation_table.candidate == "L0", "gold_macro_auc"].iloc[0]
evaluation_table["gold_vetoed"] = (evaluation_table.gold_macro_auc < l0_gold - 0.02) | evaluation_table.gold_macro_auc.isna()
evaluation_table["cost_usd"] = evaluation_table.candidate.map({cid: 0.0 for cid in CANDIDATE_IDS}).fillna(0.0)
evaluation_table.loc[evaluation_table.candidate == "L7", "cost_usd"] = l7_usage["cost_usd"]
evaluation_table["hallucinations"] = 0
evaluation_table.loc[evaluation_table.candidate == "L7", "hallucinations"] = l7_hallucination_count
per_label_dev = pd.concat(per_label_frames, ignore_index=True) if per_label_frames else pd.DataFrame()
display(coverage_table.sort_values("candidate").style.format({"coverage_pct": "{:.2f}%"}))
display(evaluation_table.sort_values("dev_macro_auc", ascending=False).style.format({
    "dev_macro_auc": "{:.4f}", "gold_macro_auc": "{:.4f}", "coverage_pct": "{:.2f}%", "cost_usd": "${:.4f}"}))
if dev_available:
    display(per_label_dev.pivot(index="label", columns="candidate", values="auc").style.format("{:.4f}"))
    rare = per_label_dev[(per_label_dev.candidate == "L0") & ~per_label_dev.in_macro][["label", "n_positive"]]
    if not rare.empty:
        print("Etiquetas con <8 positivos o AUC no calculable (fuera de macro):")
        display(rare)

In [ ]:
# 5 · Elegir ganador y escribir report_labels_v2.csv solo con dev disponible
selection_log = []
winner_id = selected_id = None
decision_auto = "pending_dev"
dev_delta_vs_l0 = np.nan

def choose_with_ties_and_gold_veto(table):
    pool = table[(table.coverage_pct >= 99.0) & table.dev_macro_auc.notna()].copy()
    while not pool.empty:
        best_auc = float(pool.dev_macro_auc.max())
        tied = pool[(best_auc - pool.dev_macro_auc) < 0.005].copy()
        candidate = tied.sort_values(["cost_usd", "dev_macro_auc", "candidate"],
                                     ascending=[True, False, True]).iloc[0]
        candidate_id = candidate.candidate
        if bool(candidate.gold_vetoed):
            selection_log.append(f"{candidate_id} vetado por gold")
            pool = pool[pool.candidate != candidate_id]
            continue
        return candidate_id
    raise RuntimeError("Ningún candidato con cobertura >=99% pasó el veto gold")

if dev_available:
    winner_id = choose_with_ties_and_gold_veto(evaluation_table)
    dev_l0 = float(evaluation_table.loc[evaluation_table.candidate == "L0", "dev_macro_auc"].iloc[0])
    dev_winner = float(evaluation_table.loc[evaluation_table.candidate == winner_id, "dev_macro_auc"].iloc[0])
    dev_delta_vs_l0 = dev_winner - dev_l0
    decision_auto = "worked" if dev_delta_vs_l0 >= 0.02 else "didnt"
    selected_id = winner_id if decision_auto == "worked" else "L0"
    selection_log.append(f"ganador dev tras vetos={winner_id}; delta vs L0={dev_delta_vs_l0:+.4f}")
    if decision_auto == "didnt":
        selection_log.append("mejora < +0.02: v1 continúa como etiquetas v2")
    selected = candidates[selected_id].set_index("StudyInstanceUID")[LABELS].reindex(train["StudyInstanceUID"])
    selected.index = train["StudyInstanceUID"].values
    selected.index.name = "StudyInstanceUID"
    output = selected.reset_index()
    output["source"] = f"L8:{'+'.join(l8_members)}" if selected_id == "L8" else selected_id
    if len(output) != len(train) or output.StudyInstanceUID.duplicated().any():
        raise RuntimeError("La salida v2 no conserva exactamente los estudios de train.csv")
    output.to_csv(OUTPUT_PATH, index=False)
    print(f"Escrito: {OUTPUT_PATH} ({len(output):,} filas, source={output.source.iloc[0]})")
    for line in selection_log:
        print(line)
else:
    print("dev pendiente: no se elige ganador ni se escribe report_labels_v2.csv")

In [ ]:
# 6 · Guardar metrics.json
def json_records(frame):
    return json.loads(frame.to_json(orient="records")) if not frame.empty else []

metrics = {
    "exp": EXP,
    "parent": PADRE,
    "phase": "smoke" if SMOKE else "run",
    "seed": SEED,
    "openai_model": OPENAI_MODEL,
    "openai_temperature_sent": 0 if model_supports_temperature_zero(OPENAI_MODEL) else None,
    "dev_available": dev_available,
    "dev_schema": dev_schema,
    "n_train": len(train),
    "n_dev": len(dev) if dev_available else 0,
    "n_gold": len(gold),
    "coverage": json_records(coverage_table.sort_values("candidate")),
    "candidate_errors": candidate_errors,
    "candidates": json_records(evaluation_table.sort_values("candidate")),
    "dev_per_label": json_records(per_label_dev),
    "labels_excluded_from_dev_macro": (per_label_dev[(per_label_dev.candidate == "L0") & ~per_label_dev.in_macro]["label"].tolist()
                                               if not per_label_dev.empty else []),
    "l7": {
        "processed": len(l7_records), "errors": len(l7_errors),
        "error_details": l7_errors, "pending_without_key": l7_pending_without_key,
        "hallucinations": l7_hallucination_count,
        "hallucination_rate": (float(l7_hallucination_rate) if np.isfinite(l7_hallucination_rate) else None),
        "usage": l7_usage, "max_cost_usd": MAX_API_COST_USD,
    },
    "l8_members": l8_members,
    "winner": winner_id,
    "selected_for_v2": selected_id,
    "dev_delta_vs_l0": (float(dev_delta_vs_l0) if np.isfinite(dev_delta_vs_l0) else None),
    "decision_auto": decision_auto,
    "selection_log": selection_log,
    "output_path": str(OUTPUT_PATH) if dev_available else None,
}
metrics_path = guardar_metricas(RESULTS, metrics)
print("Guardado:", metrics_path)

# 7 · Verdict

Después del run real (sin dev sintético y con `SMOKE = False`):

1. Copiar **solo** los números de `results/exp011_etiquetas_v2/metrics.json` a la sección 3 de `SPEC.md`.
2. Completar la fila de `docs/EXPERIMENTOS.md` con `worked`, `didnt` o `killed`.
3. Confirmar que gold únicamente vetó candidatos y nunca alteró el ranking del dev.
4. No subir caches GPT, datasets públicos, `report_labels_v2.csv` ni ningún dato al repo.